# Stage 3 — Model Development & Tracking

In [3]:
import sys
import importlib

sys.path.insert(0, ".")

import data_prep as dp
import config as cfg

importlib.reload(dp)
importlib.reload(cfg)

print("data_prep loaded:", hasattr(dp, "prepare"))
print("config loaded:", cfg.DATA_PATH)

data_prep loaded: True
config loaded: diabetic_data.csv


In [1]:
import config as cfg

print("Config loaded from:")
print(cfg.__file__)

print("\nDATA_PATH:", cfg.DATA_PATH)
print("RANDOM_STATE:", cfg.RANDOM_STATE)

Config loaded from:
c:\Users\368435\Music\MlProjects\hospital-readmission-prediction\config.py

DATA_PATH: diabetic_data.csv
RANDOM_STATE: 42


In [4]:
X, y, X_train, X_val, X_test, y_train, y_val, y_test = dp.prepare(
    cfg.DATA_PATH
)

print("========================================")
print("Stage 3 Data Preparation Validation")
print("========================================")

print("Full dataset:", X.shape)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nPositive rates:")
print("Full:", round(y.mean() * 100, 2), "%")
print("Train:", round(y_train.mean() * 100, 2), "%")
print("Validation:", round(y_val.mean() * 100, 2), "%")
print("Test:", round(y_test.mean() * 100, 2), "%")

c:\Users\368435\Music\MlProjects\hospital-readmission-prediction\data_prep.py:75: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, na_values=["?"])


Stage 3 Data Preparation Validation
Full dataset: (69973, 54)
Train: (48981, 54)
Validation: (10496, 54)
Test: (10496, 54)

Target distribution:
target
0    63696
1     6277
Name: count, dtype: int64

Positive rates:
Full: 8.97 %
Train: 8.97 %
Validation: 8.97 %
Test: 8.97 %


In [6]:
print("========================================")
print("Preprocessor Validation")
print("========================================")

print("Preprocessor type:")
print(type(preprocessor))

print("\nTransformer names:")
print([name for name, _, _ in preprocessor.transformers])

print("\nNumber of transformers:")
print(len(preprocessor.transformers))

Preprocessor Validation
Preprocessor type:
<class 'sklearn.compose._column_transformer.ColumnTransformer'>

Transformer names:
['numeric', 'categorical']

Number of transformers:
2


In [7]:
numeric_features, categorical_features = dp.get_feature_lists(X_train)

print("========================================")
print("Feature Coverage Validation")
print("========================================")

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numeric_features) + len(categorical_features))

print("\nNumeric features:")
print(numeric_features)

print("\nExcluded raw features:")
excluded_raw = [
    "age",
    "diag_1",
    "diag_2",
    "diag_3",
    "medical_specialty"
]

print([
    col for col in excluded_raw
    if col in X_train.columns
])

print("\nTarget/ID leakage check:")
leakage_columns = [
    "target",
    "encounter_id",
    "patient_nbr",
    "readmitted"
]

print([
    col for col in leakage_columns
    if col in numeric_features + categorical_features
])

Feature Coverage Validation
Numeric features: 11
Categorical features: 35
Total features: 46

Numeric features:
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_midpoint', 'total_prior_visits', 'medication_change_count']

Excluded raw features:
['age', 'diag_1', 'diag_2', 'diag_3', 'medical_specialty']

Target/ID leakage check:
[]


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_preprocessor = dp.build_preprocessor(X_train)

baseline_model = Pipeline([
    ("preprocessor", baseline_preprocessor),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=cfg.RANDOM_STATE
        )
    )
])

print("========================================")
print("Logistic Regression Baseline")
print("========================================")

print("Pipeline created successfully")
print("Steps:", list(baseline_model.named_steps.keys()))

Logistic Regression Baseline
Pipeline created successfully
Steps: ['preprocessor', 'classifier']


In [9]:
print("========================================")
print("Training Logistic Regression Baseline")
print("========================================")

baseline_model.fit(X_train, y_train)

print("Baseline model training completed successfully.")

Training Logistic Regression Baseline
Baseline model training completed successfully.


In [10]:
print("========================================")
print("Generating Validation Predictions")
print("========================================")

y_val_proba = baseline_model.predict_proba(X_val)[:, 1]
y_val_pred = baseline_model.predict(X_val)

print("Validation predictions generated successfully.")

print("\nPrediction shapes:")
print("Probability predictions:", y_val_proba.shape)
print("Class predictions:", y_val_pred.shape)

print("\nFirst 10 predicted probabilities:")
print(y_val_proba[:10])

print("\nFirst 10 predicted classes:")
print(y_val_pred[:10])

Generating Validation Predictions
Validation predictions generated successfully.

Prediction shapes:
Probability predictions: (10496,)
Class predictions: (10496,)

First 10 predicted probabilities:
[0.61199846 0.47634658 0.69320518 0.43032186 0.45762295 0.61220296
 0.60404742 0.34178658 0.38868058 0.41100296]

First 10 predicted classes:
[1 0 1 0 0 1 1 0 0 0]


In [11]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score
)

print("========================================")
print("Logistic Regression - Validation Metrics")
print("========================================")

baseline_roc_auc = roc_auc_score(y_val, y_val_proba)
baseline_pr_auc = average_precision_score(y_val, y_val_proba)
baseline_recall = recall_score(y_val, y_val_pred)
baseline_precision = precision_score(y_val, y_val_pred)
baseline_f1 = f1_score(y_val, y_val_pred)
baseline_accuracy = accuracy_score(y_val, y_val_pred)

print(f"ROC-AUC  : {baseline_roc_auc:.4f}")
print(f"PR-AUC   : {baseline_pr_auc:.4f}")
print(f"Recall   : {baseline_recall:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"F1 Score : {baseline_f1:.4f}")
print(f"Accuracy : {baseline_accuracy:.4f}")

Logistic Regression - Validation Metrics
ROC-AUC  : 0.6415
PR-AUC   : 0.1705
Recall   : 0.5276
Precision: 0.1359
F1 Score : 0.2162
Accuracy : 0.6566
